# Theoretical SSA evolution curves

This notebook compares modeled snow SSA evolution under isothermal and temperature-gradient conditions. It evaluates the B21 and B25 parameterizations and explores how the B25 temperature-gradient, curvature, and diffusion-length terms affect the predicted rate of change.

## Workflow
1. Define helper functions for evolving SSA or optical diameter through time.
2. Compare B21 and B25 predictions across example temperatures and gradients.
3. Calculate the component contributions in the B25 formulation and plot parameter-sensitivity experiments.

The model functions are imported from `function_SSA.py`. Figures are saved to paths specified in the plotting cells; update those paths if needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#local import
import function_SSA as core_ssa

In [ ]:
def evolve_ssa_B25(time_array, ssa_0, temperature, grad, density, dt, a_A = 1320, a_B = 1.52e-2, a_L = 1.09e-2, ssa_minimum = 0):
    ssa_l = [ssa_0]
    ssa = ssa_0
    for time in time_array:
        if time == 0:
            continue
        else:
            ssa = core_ssa.ssa_B25(ssa, temperature, grad, density, dt, a_A = a_A, a_B = a_B, a_L = a_L, ssa_minimum = ssa_minimum)
            ssa_l.append(ssa)

    return np.array(ssa_l)

def B21_calculate_do(temperature, gradient, density, s_0, do_0, days, og = False, exponant = 0.4):
    #dt in second
    dt = days[1]-days[0]
    do_toy = [do_0]
    for day in days:
        if day == 0 :
            s_f = np.clip(core_ssa.grad_s(temperature + 273.15, gradient, exponant = exponant) * dt *3600+ s_0, a_min = 0, a_max = 1)
            # s_f = np.max(core_ssa.grad_s(temperature + 273.15, gradient, exponant = exponant) * dt *3600+ s_0, 0)
            do = core_ssa.grad_d(temperature + 273.15, gradient, density, s_f, do_0, og, exponant = exponant)* dt *3600+ do_0
            do_toy.append(do)

        elif day == days[-1]:
            continue
        elif s_f == 1:
            # print(s_f)
            do = core_ssa.grad_d(temperature + 273.15, gradient, density, s_f, do, og, exponant = exponant)* dt *3600+ do
            do_toy.append(do)

        else:
            s_f = np.clip(core_ssa.grad_s(temperature + 273.15, gradient, exponant = exponant)* dt *3600+ s_f, a_min = 0, a_max = 1)
            # s_f = np.max(core_ssa.grad_s(temperature + 273.15, gradient, exponant = exponant) * dt *3600+ s_f, 0)
            # print(s_f)
            # print(core_ssa.grad_d(temperature + 273.15, gradient, density, s_f, do, og, exponant = exponant))
            do = core_ssa.grad_d(temperature + 273.15, gradient, density, s_f, do, og, exponant = exponant)* dt *3600+ do
            do_toy.append(do)
    return np.array(do_toy)

In [ ]:
s_0 = 0
do_0 = 1e-4
ssa_0 = 6/(917*do_0)
density = 200
hours = np.arange(0, 60*24)
days = np.arange(0,60)
dt =1
gradient = [1, 10, 50]
temp = np.array([-2,-10, -25])

fig, ax = plt.subplots(3,3, figsize = (6,6), sharey=True,sharex=True)


for i in range(3):
    for j in range(3):
        ax[i,j].set_title(f"T = {temp[i]}$\degree$C, $\\nabla$T = {gradient[j]} K m$^{{-1}}$", size = 9)
        do = B21_calculate_do(temp[i], gradient[j], density, 0, do_0, hours, og = True)[::24]

        ax[i,j].plot(days, 6/(917*do), label = 'B21')
        ssa = evolve_ssa_B25(days, ssa_0, temp[i] + 273.15, gradient[j], density, dt*24, 
                             a_A = 551.1,
                             a_B = 0.082, 
                             a_L = 906.5, 
                             ssa_minimum = 8)
        ax[i,j].plot(days, ssa, label = 'B25')
        ax[i,j].grid(True, linestyle="--", alpha=0.5)

ax[0,0].set_ylabel('SSA (m$^2$ kg$^{-1}$)' )
ax[1,0].set_ylabel('SSA (m$^2$ kg$^{-1}$)' )
ax[2,0].set_ylabel('SSA (m$^2$ kg$^{-1}$)' )
ax[2,0].set_xlabel('Days')
ax[2,1].set_xlabel('Days')
ax[2,2].set_xlabel('Days')
ax[0,0].legend()
fig.tight_layout()
# fig.savefig('/home/jum002/public_html/ssa/ET_TG_exemple_case.png', dpi = 300)



In [ ]:
#constant
D_v = 2e-5  #vapor diffusion coefficient
v_kin = 140  #kinetic velocity
c_l =  1e-2  #capillary length
rho_ice = 916.7 # density of ice
a_L = 1e-2
s_i= 40*917
alpha = 1e-1

#transition coefficient between kinetic limited to diffusion limited
1/(1 + D_v / (alpha * v_kin * a_L / s_i))

In [ ]:
def ds_dt_B25(ssa_i, T_mid, T_grad, density, dt, a_A = 1320, a_B = 0.2, a_L = 1e-2, ssa_minimum = 0, alpha = 1e-1):
    """
    SSA evolution from Braun et al 2025
    doi:10.1017/jog.2024.109
    """

    #param fit 
    # a_A # sensitivity to temperature gradient
    # a_B # sensitivity to curvature
    # a_L # length of diffusion
    # ssa_minimum # ssa minimum value
    s_minimum = ssa_minimum * density
    s_i = ssa_i * density

    #constant
    D_v = 2e-5  #vapor diffusion coefficient
    #v_kin = 140  #kinetic velocity
    c_l =  1e-9  #capillary length
    rho_ice = 916.7 # density of ice
    kb = 1.380649e-23
    m_water = 2.99e-26 #kg , mass of water molecules

    v_kin = np.sqrt(kb * T_mid / (2 * np.pi * m_water))
    
    #convert dt in seconds
    dt_s = dt * 3600

    #temperature gradient term
    TG_contribution = a_A * beta(T_mid) * T_grad

    sigma_H = (ssa_i * rho_ice)**2

    #isothermal term
    isothermal_contribution = a_B * sigma_H**1.5 * compute_rho_vs(T_mid) * c_l

    #transition coefficient between kinetic limited to diffusion limited
    KL_DL_transition = 1 + D_v / (alpha * v_kin * a_L / s_i)

    #print(-2 * D_v / rho_ice, TG_contribution, isothermal_contribution, KL_DL_transition)
    #print(-2 * D_v / rho_ice, (TG_contribution + isothermal_contribution)/ KL_DL_transition)

    # variation of ssa over time
    ds_dt =  - 2 * s_i * D_v / rho_ice * (TG_contribution + isothermal_contribution )/ KL_DL_transition
    ds_dt_iso =  - 2 * s_i * D_v / rho_ice * (isothermal_contribution )/ KL_DL_transition
    ds_dt_TG =  - 2 * s_i*  D_v / rho_ice * (TG_contribution)/ KL_DL_transition

    #print(D_v /rho_ice, TG_contribution, isothermal_contribution, KL_DL_transition)
    #print(dssa_dt)

    return ds_dt * 3600*24/density, ds_dt_iso * 3600*24/density, ds_dt_TG * 3600*24/density

def compute_rho_vs(T_mid, derivative = False):
    #constant
    m_water = 2.99e-26 #kg , mass of water molecules
    kb = 1.380649e-23 #boltzman constant
    P_0 = 611 # Pa
    T_0 = 273.15 #K
    L_fus = 8.37e-20 # J
    if derivative:
        return m_water/(kb * T_mid**3) * P_0 * (L_fus/kb - T_mid) * np.exp(L_fus/kb * (1/T_0 - 1/T_mid))
    else:
        return m_water/(kb * T_mid) * P_0 * np.exp(L_fus/kb * (1/T_0 - 1/T_mid))

def beta(T_mid):
    return compute_rho_vs(T_mid, derivative = True)

In [ ]:
ssa_a = np.linspace(0,60,10)

#a_B
ds_dt_0_iso_b1 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.01, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_0_iso_b2 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.03, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_0_iso_b3 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_0_TG_b = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.07, a_L = 2e-2)[2] for ssa in ssa_a]

ds_dt_5_iso_b1 = [ds_dt_B25(ssa, 270, 5, 250, dt, a_A = 924, a_B = 0.01, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_5_iso_b2 = [ds_dt_B25(ssa, 270, 5, 250, dt, a_A = 924, a_B = 0.03, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_5_iso_b3 = [ds_dt_B25(ssa, 270, 5, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_5_TG_b = [ds_dt_B25(ssa, 270, 5, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]

ds_dt_50_iso_b1 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.01, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_50_iso_b2 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.03, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_50_iso_b3 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_50_TG_b = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]

#a_A
ds_dt_0_iso_a = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_0_TG_a1 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 500, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_0_TG_a2 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 100, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_0_TG_a3 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 1500, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]

ds_dt_15_iso_a = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_15_TG_a1 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 500, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_15_TG_a2 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 1000, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_15_TG_a3 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 1500, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]

ds_dt_50_iso_a = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_50_TG_a1 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 500, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_50_TG_a2 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 1000, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_50_TG_a3 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 1500, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]

#a_L
ds_dt_0_iso_l1 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_0_iso_l2 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e2)[1] for ssa in ssa_a]
ds_dt_0_iso_l3 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e0)[1] for ssa in ssa_a]
ds_dt_0_TG_l1 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_0_TG_l2 = [ds_dt_B25(ssa, 270, 0, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e2)[2] for ssa in ssa_a]

ds_dt_15_iso_l1 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_15_iso_l2 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 924, a_B = 0.05, a_L = 2)[1] for ssa in ssa_a]
# ds_dt_15_iso_l3 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e0)[1] for ssa in ssa_a]
ds_dt_15_TG_l1 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_15_TG_l2 = [ds_dt_B25(ssa, 270, 15, 250, dt, a_A = 924, a_B = 0.05, a_L = 2)[2] for ssa in ssa_a]

ds_dt_50_iso_l1 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[1] for ssa in ssa_a]
ds_dt_50_iso_l2 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2)[1] for ssa in ssa_a]
# ds_dt_50_iso_l3 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e0)[1] for ssa in ssa_a]
ds_dt_50_TG_l1 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2e-2)[2] for ssa in ssa_a]
ds_dt_50_TG_l2 = [ds_dt_B25(ssa, 270, 50, 250, dt, a_A = 924, a_B = 0.05, a_L = 2)[2] for ssa in ssa_a]



In [ ]:
fig, ax = plt.subplots(3,3, figsize = (10,8), sharey = True, sharex = True)


#a_B
ax[0,0].plot(ssa_a, ds_dt_0_iso_b1, label = 'iso, a_B = 0.01', linestyle = ':', color = 'darkorange')
ax[0,0].plot(ssa_a, ds_dt_0_iso_b2, label = 'iso, a_B = 0.03', linestyle = '--', color = 'darkorange')
ax[0,0].plot(ssa_a, ds_dt_0_iso_b3, label = 'iso, a_B = 0.05', color = 'darkorange')
ax[0,0].plot(ssa_a, ds_dt_0_TG_b, color = 'dodgerblue')
ax[0,0].legend()
ax[0,0].set_title('$\\nabla T$ = 0')


ax[0,1].plot(ssa_a, ds_dt_5_iso_b1, label = 'iso, a_b = 0.01', linestyle = ':', color = 'darkorange')
ax[0,1].plot(ssa_a, ds_dt_5_iso_b2, label = 'iso, a_b = 0.03', linestyle = '--', color = 'darkorange')
ax[0,1].plot(ssa_a, ds_dt_5_iso_b3, label = 'iso, a_b = 0.05', color = 'darkorange')
ax[0,1].plot(ssa_a, ds_dt_5_TG_b, label = 'TG', color = 'dodgerblue')
ax[0,1].set_title('$\\nabla T$ = 5')


ax[0,2].plot(ssa_a, ds_dt_50_iso_b1, label = 'iso, a_b = 0.01', linestyle = ':', color = 'darkorange')
ax[0,2].plot(ssa_a, ds_dt_50_iso_b2, label = 'iso, a_b = 0.03', linestyle = '--', color = 'darkorange')
ax[0,2].plot(ssa_a, ds_dt_50_iso_b3, label = 'iso, a_b = 0.05', color = 'darkorange')
ax[0,2].plot(ssa_a, ds_dt_50_TG_b, label = 'TG', color = 'dodgerblue')
ax[0,2].set_title('$\\nabla T$ = 50')

#a_A
ax[1,0].plot(ssa_a, ds_dt_0_iso_a, color = 'darkorange')
ax[1,0].plot(ssa_a, ds_dt_0_TG_a1, label = 'TG, a_A = 500', linestyle = ':', color = 'dodgerblue')
ax[1,0].plot(ssa_a, ds_dt_0_TG_a2, label = 'TG, a_A = 1000', linestyle = '--', color = 'dodgerblue')
ax[1,0].plot(ssa_a, ds_dt_0_TG_a3, label = 'TG, a_A = 1500', color = 'dodgerblue')
ax[1,0].legend()
ax[1,0].set_title('$\\nabla T$ = 0')


ax[1,1].plot(ssa_a, ds_dt_15_iso_a, label = 'iso, a_b = 0.05', color = 'darkorange')
ax[1,1].plot(ssa_a, ds_dt_15_TG_a1, label = 'TG, a_A = 500', linestyle = ':', color = 'dodgerblue')
ax[1,1].plot(ssa_a, ds_dt_15_TG_a2, label = 'TG, a_A = 1000', linestyle = '--', color = 'dodgerblue')
ax[1,1].plot(ssa_a, ds_dt_15_TG_a3, label = 'TG, a_A = 1500', color = 'dodgerblue')
ax[1,1].set_title('$\\nabla T$ = 15')


ax[1,2].plot(ssa_a, ds_dt_50_iso_a, label = 'iso, a_b = 0.05', color = 'darkorange')
ax[1,2].plot(ssa_a, ds_dt_50_TG_a1, label = 'TG, a_A = 500', linestyle = ':', color = 'dodgerblue')
ax[1,2].plot(ssa_a, ds_dt_50_TG_a2, label = 'TG, a_A = 1000', linestyle = '--', color = 'dodgerblue')
ax[1,2].plot(ssa_a, ds_dt_50_TG_a3, label = 'TG, a_A = 1500', color = 'dodgerblue')
ax[1,2].set_title('$\\nabla T$ = 50')

#a_L
ax[2,0].plot(ssa_a, ds_dt_0_iso_l1, label = 'iso, a_L = $2x10^{-3}$', color = 'darkorange')
ax[2,0].plot(ssa_a, ds_dt_0_iso_l2, label = 'iso, a_L = $2x10^{3}$', linestyle = ':', color = 'darkorange')
# ax[2,0].plot(ssa_a, ds_dt_0_iso_l3, label = 'iso, a_L = 1', linestyle = ':')
ax[2,0].plot(ssa_a, ds_dt_0_TG_l1, label = 'TG, a_L = $2x10^{-3}$', color = 'dodgerblue')
ax[2,0].plot(ssa_a, ds_dt_0_TG_l2, label = 'TG, a_L = $2x10^{3}$', linestyle = ':', color = 'dodgerblue')
ax[2,0].legend()
ax[2,0].set_title('$\\nabla T$ = 0')


ax[2,1].plot(ssa_a, ds_dt_15_iso_l1, label = 'iso, a_L = 0.017', color = 'darkorange')
ax[2,1].plot(ssa_a, ds_dt_15_iso_l2, label = 'iso, a_L = 1700', linestyle = ':', color = 'darkorange')
# ax[2,1].plot(ssa_a, ds_dt_15_iso_l3, label = 'iso, a_L = 1', linestyle = ':')
ax[2,1].plot(ssa_a, ds_dt_15_TG_l1, color = 'dodgerblue')
ax[2,1].plot(ssa_a, ds_dt_15_TG_l2, linestyle = ':', color = 'dodgerblue')
ax[2,1].set_title('$\\nabla T$ = 15')


ax[2,2].plot(ssa_a, ds_dt_50_iso_l1, label = 'iso, a_L = 0.017', color = 'darkorange')
ax[2,2].plot(ssa_a, ds_dt_50_iso_l2, label = 'iso, a_L = 1700', color = 'darkorange', linestyle = ':')
# ax[2,2].plot(ssa_a, ds_dt_50_iso_l3, label = 'iso, a_L = 1', linestyle = ':')
ax[2,2].plot(ssa_a, ds_dt_50_TG_l1, color = 'dodgerblue')
ax[2,2].plot(ssa_a, ds_dt_50_TG_l2, linestyle = ':', color = 'dodgerblue')
ax[2,2].set_title('$\\nabla T$ = 50')


# ax[1,0].plot(ssa_a, ds_dt_0_total, label = 'total', linestyle = '--', zorder= 10)
# ax[1,0].plot(ssa_a, ds_dt_0_iso, label = 'iso')
# ax[1,0].plot(ssa_a, ds_dt_0_TG, label = 'TG')


# ax[1,1].plot(ssa_a, ds_dt_50_total, label = 'total', linestyle = '--')
# ax[1,1].plot(ssa_a, ds_dt_5_iso, label = 'iso')
# ax[1,1].plot(ssa_a, ds_dt_5_TG, label = 'TG')



ax[2,0].set_xlabel('$SSA_M$')
ax[2,1].set_xlabel('$SSA_M$')
ax[2,2].set_xlabel('$SSA_M$')
ax[0,0].set_ylabel('$d_{SSA}/dt$')
ax[1,0].set_ylabel('$d_{SSA}/dt$')
ax[2,0].set_ylabel('$d_{SSA}/dt$')
fig.tight_layout()

fig.savefig('/home/jum002/public_html/ssa/dsdt_exemple_param.png', dpi = 300)